In [0]:
tabelas = {
    "bronze": [
        "clientes", "contas", "cartoes",
        "transacoes", "estornos"
    ],
    "prata": [
        "clientes", "contas", "cartoes",
        "transacoes", "estornos", "quarentena"
    ],
    "ouro": [
        "gold_transacoes_cartao",
        "gold_resumo_cliente_mes",
        "gold_ranking_estabelecimento",
        "gold_alertas_qualidade"
    ]
}

resultado = []

for camada, nomes in tabelas.items():
    for nome in nomes:
        tabela = f"novarota.{camada}.{nome}"

        quantidade = spark.table(tabela).count()

        resultado.append(
            (camada, nome, quantidade)
        )

df_resultado = spark.createDataFrame(
    resultado,
    ["camada", "tabela", "quantidade"]
)

display(df_resultado.orderBy("camada", "tabela"))

In [0]:
%sql
SELECT
    arquivo_origem,
    data_referencia,
    COUNT(*) AS total_registros
FROM novarota.bronze.transacoes
GROUP BY
    arquivo_origem,
    data_referencia
ORDER BY data_referencia;

In [0]:
%sql
SELECT
    entidade,
    tipo_erro,
    SUM(qtd_rejeitados) AS total_rejeitados
FROM novarota.ouro.gold_alertas_qualidade
GROUP BY entidade, tipo_erro
ORDER BY total_rejeitados DESC;

In [0]:
display(
    spark.table("novarota.prata.quarentena")
)

In [0]:
%sql
SELECT
    id_transacao,
    COUNT(*) AS quantidade
FROM novarota.prata.transacoes
GROUP BY id_transacao
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    id_transacao,
    id_cartao,
    valor,
    estornada
FROM novarota.prata.transacoes
WHERE estornada = TRUE;

In [0]:
%sql
SELECT
    COUNT(*) AS estornos_incorretos_na_ouro
FROM novarota.ouro.gold_transacoes_cartao g
INNER JOIN novarota.prata.transacoes p
    ON g.id_transacao = p.id_transacao
WHERE p.estornada = TRUE;

In [0]:
%sql
SELECT
    (SELECT COUNT(*)
     FROM novarota.ouro.gold_transacoes_cartao)
        AS quantidade_transacoes,

    (SELECT SUM(qtd_transacoes)
     FROM novarota.ouro.gold_resumo_cliente_mes)
        AS quantidade_resumo,

    (SELECT SUM(valor)
     FROM novarota.ouro.gold_transacoes_cartao)
        AS valor_transacoes,

    (SELECT SUM(valor_total)
     FROM novarota.ouro.gold_resumo_cliente_mes)
        AS valor_resumo;

In [0]:
%sql
WITH base AS (
  SELECT id_cliente, nome_cliente, ano_mes,
         TO_DATE(CONCAT(ano_mes, '-01')) AS mes,
         valor_total
  FROM novarota.ouro.gold_resumo_cliente_mes
),
movimentacao AS (
  SELECT *,
         LAG(valor_total) OVER (PARTITION BY id_cliente ORDER BY mes) AS valor_mes_anterior,
         LAG(mes) OVER (PARTITION BY id_cliente ORDER BY mes) AS mes_anterior
  FROM base
)
SELECT id_cliente, nome_cliente, ano_mes,
       valor_mes_anterior, valor_total,
       ROUND(100 * (valor_total - valor_mes_anterior) / valor_mes_anterior, 2) AS crescimento_percentual
FROM movimentacao
WHERE mes_anterior = ADD_MONTHS(mes, -1)
  AND valor_mes_anterior > 0
  AND valor_total > valor_mes_anterior * 1.50
ORDER BY crescimento_percentual DESC;

In [0]:
%sql
WITH base AS (
  SELECT id_cliente, MAX(nome_cliente) AS nome_cliente,
         SUM(valor_total) AS total_movimentado
  FROM novarota.ouro.gold_resumo_cliente_mes
  GROUP BY id_cliente
), quartis AS (
  SELECT *, NTILE(4) OVER (ORDER BY total_movimentado DESC, id_cliente) AS quartil
  FROM base
)
SELECT * FROM quartis ORDER BY quartil, total_movimentado DESC;

In [0]:
%sql
WITH resumo AS (
  SELECT ano_mes, cidade_estabelecimento, nome_estabelecimento,
         SUM(valor) AS valor_total,
         COUNT(*) AS qtd_transacoes
  FROM novarota.ouro.gold_transacoes_cartao
  GROUP BY ano_mes, cidade_estabelecimento, nome_estabelecimento
), ranking AS (
  SELECT *, DENSE_RANK() OVER (
    PARTITION BY ano_mes, cidade_estabelecimento
    ORDER BY valor_total DESC, qtd_transacoes DESC
  ) AS posicao
  FROM resumo
)
SELECT * FROM ranking ORDER BY ano_mes, cidade_estabelecimento, posicao;


In [0]:
%sql
SELECT COUNT(*) AS transacoes_validas,
       SUM(CAST(estornada AS INT)) AS transacoes_estornadas
FROM novarota.prata.transacoes;



In [0]:
%sql
SELECT entidade, tipo_erro, SUM(qtd_rejeitados) AS total
FROM novarota.ouro.gold_alertas_qualidade
GROUP BY entidade, tipo_erro
ORDER BY total DESC;